# Calculating the Implied Earnings Move Using Options Prices

Earnings day can be a pivotal moment for a company's share price. The confluence of expectations and reality is a tradable event, drawing crowds to the options market. Observing the surrounding action can provide insight into the consensus view on, and general sentiment of, the company.

The cost of a straddle - the combined price of an at-the-money call and put - is a common way to gauge the near-term volatility. It's the market's expectation of the price band until expiration. While this includes time value, the isolated price of volatility will generally be higher for the expiry immediately following an earnings release.

Have a look at companies that trade weekly options and are reporting a on Thursday. If they report after the close, the price of the one-day straddle at the bell will be the purest sample of information.

The cells below will demonstrate how to get the data from free sources, using the OpenBB Platform.

In [ ]:
# If using in Google Colab, install the OpenBB library.

#!pip install openbb["all"]

# Restart the runtime before the next block

In [ ]:
from datetime import datetime, timedelta

from openbb import obb

obb.user.preferences.output_type = "dataframe"

If the earnings date falls on an option expiry, contracts expiring that day will not provide exposure to the after-market earnings reports.

In [ ]:
# Lookup some upcoming earnings dates and sort them by market cap.

earnings_calendar = obb.equity.calendar.earnings(
    start_date=(datetime.now() + timedelta(days=1)).date(),
    end_date=(datetime.now() + timedelta(days=14)).date(),
    provider="nasdaq",
)

earnings_calendar.sort_values(by=["market_cap", "num_estimates"], ascending=False).head(
    20
)

In [ ]:
# Get the options chains data.

symbol = "NVDA"  # This will not be evergreen, change the symbol based on a stock above.

obb.user.preferences.output_type = "OBBject"  # To use the built-in options chains methods, we need to set the output type to OBBject.

options = obb.derivatives.options.chains(symbol, provider="cboe")

last_price = options.results.underlying_price[0]

display(f"Last Price: ${last_price}")

display(options.results.expirations[:3])

In [ ]:
# Use the straddle method of the results object to get the straddle data and then calculate the implied move.

straddle = options.results.straddle(days=options.results.expirations[1])
straddle_price = straddle.loc["Cost"].values[0]
days = straddle.loc["DTE"].values[0]
upper_price = straddle.loc["Breakeven Upper"].values[0]
lower_price = straddle.loc["Breakeven Lower"].values[0]

implied_move = ((1 + straddle_price / last_price) ** (1 / days) - 1) * 100

display(
    f"Cost of Straddle: ${round(straddle_price, 2)}"
    f"\nCost as a % of Share Price: {round((straddle_price/last_price) * 100, 4)}%"
    f"\nUpper Breakeven Price: ${upper_price}"
    f"\nLower Breakeven Price: ${lower_price}"
    f"\nImplied Daily Move: {round(implied_move, 4)}%\n"
)

display(straddle)